In [1]:
from pathlib import Path
import hashlib
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display

warnings.filterwarnings('ignore')

ROOT = Path('.')
TRAIN_PATH = ROOT / 'train.csv'
TEST_PATH = ROOT / 'test.csv'
SAMPLE_PATH = ROOT / 'sample_submission.csv'
PRIMARY_SUBMISSION_PATH = ROOT / 'sample_submission.csv'
SEED = 322

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'CUDA device count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f'GPU memory (GB): {total_gb:.2f}')
    bf16_ok = hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported()
    print(f'BF16 supported: {bf16_ok}')
    # speed knobs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
else:
    print('GPU not detected. Pipeline will fall back where possible, but full mode expects CUDA.')

for path in [TRAIN_PATH, TEST_PATH, SAMPLE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path.resolve()}')

Python: 3.13.2
Platform: macOS-26.4.1-arm64-arm-64bit-Mach-O
Torch: 2.12.0
CUDA available: False
CUDA device count: 0
GPU not detected. Pipeline will fall back where possible, but full mode expects CUDA.


In [ ]:


from __future__ import annotations

import ast
import contextlib
import copy
import gc
import hashlib
import html
import io
import json
import os
import random
import re
import shutil
import tempfile
import time
from collections import Counter
from typing import Any

import numpy as np
import pandas as pd
import torch
from scipy import sparse
from scipy.optimize import minimize
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    MultilabelStratifiedKFold = None
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import hamming_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import normalize
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False




TAG_RE = re.compile(r"<[^>]+>")
CDATA_RE = re.compile(r"<!\[CDATA\[.*?\]\]>", flags=re.S)
ENTITY_RE = re.compile(r"&[#A-Za-z0-9]+;")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
NON_TEXT_RE = re.compile(r"[^0-9a-zа-я\s.,!?;:%()\-]+")
TRANSFORMER_NON_TEXT_RE = re.compile(r"[^0-9A-Za-zА-Яа-яЁё\s.,!?;:%()\-\"'«»:/]+")
SPACE_RE = re.compile(r"\s+")
TOKEN_RE = re.compile(r"[0-9a-zа-я]+")



def set_seed(seed: int = 322) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(device: str = "cpu") -> torch.device:
    if device != "auto":
        return torch.device(device)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def pick_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    if hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def hamming_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return 1.0 - hamming_loss(y_true, y_pred)




def clean_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = html.unescape(text)
    text = CDATA_RE.sub(" ", text)
    text = TAG_RE.sub(" ", text)
    text = ENTITY_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = text.replace("ё", "е").lower()
    text = NON_TEXT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


def clean_text_for_transformer(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = html.unescape(text)
    text = CDATA_RE.sub(" ", text)
    text = TAG_RE.sub(" ", text)
    text = ENTITY_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = text.replace("\xa0", " ")
    text = text.replace("“", '"').replace("”", '"').replace("„", '"')
    text = text.replace("’", "'").replace("`", "'")
    text = TRANSFORMER_NON_TEXT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


def prepare_frame(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.copy()
    frame["publication_dt"] = pd.to_datetime(frame["publication_date"])
    frame["title_clean"] = frame["title"].map(clean_text)
    frame["text_clean"] = frame["text"].map(clean_text)
    frame["title_transformer_clean"] = frame["title"].map(clean_text_for_transformer)
    frame["text_transformer_clean"] = frame["text"].map(clean_text_for_transformer)
    frame["year"] = frame["publication_dt"].dt.year.astype(str)
    frame["month"] = frame["publication_dt"].dt.month.astype(int)
    frame["month_str"] = frame["publication_dt"].dt.month.astype(str).str.zfill(2)
    frame["weekday"] = frame["publication_dt"].dt.weekday.astype(int)
    frame["source_token"] = "src_" + frame["source"].astype(str).str.lower()
    frame["meta_text"] = (
        frame["source_token"]
        + " year_" + frame["year"]
        + " month_" + frame["month_str"]
        + " weekday_" + frame["weekday"].astype(str)
    )
    frame["combined_text"] = (
        frame["title_clean"] + " [SEP] " + frame["title_clean"]
        + " [SEP] " + frame["text_clean"].str.slice(0, 4000)
        + " " + frame["meta_text"]
    )
    frame["plain_text"] = (
        frame["title_clean"] + " [SEP] " + frame["title_clean"]
        + " [SEP] " + frame["text_clean"].str.slice(0, 4000)
    )
    frame["plain_lead_text"] = frame["title_clean"] + " [SEP] " + frame["text_clean"].str.slice(0, 1600)
    frame["plain_title_text"] = frame["title_clean"]

    frame["transformer_text"] = (
        "источник " + frame["source"].astype(str).str.lower()
        + " месяц " + frame["month_str"]
        + " деньнедели " + frame["weekday"].astype(str)
        + " заголовок " + frame["title_transformer_clean"]
        + " текст " + frame["text_transformer_clean"].str.slice(0, 6000)
    )
    frame["plain_transformer_text"] = (
        "заголовок " + frame["title_transformer_clean"]
        + " текст " + frame["text_transformer_clean"].str.slice(0, 6000)
    )

    frame["e5_passage_text"] = (
        "passage: " + frame["title_transformer_clean"]
        + ". " + frame["text_transformer_clean"].str.slice(0, 3000)
    )
    return frame


def parse_targets(target_series: pd.Series) -> np.ndarray:
    return np.asarray(target_series.map(ast.literal_eval).tolist(), dtype=np.int64)


def format_target_list(labels: np.ndarray) -> str:
    return "[" + ",".join(map(str, labels.astype(int).tolist())) + "]"


def compute_prediction_fingerprint(predicted_labels: np.ndarray) -> str:
    encoded = "\n".join(format_target_list(row) for row in predicted_labels)
    return hashlib.md5(encoded.encode("utf-8")).hexdigest()




def build_fold_keys(y: np.ndarray, n_splits: int) -> pd.Series:
    combos = pd.Series(["".join(map(str, row.tolist())) for row in y])
    combo_counts = combos.value_counts()
    density = y.sum(axis=1).astype(int)
    keys = combos.where(combos.map(combo_counts) >= n_splits, "rare_" + pd.Series(density, index=combos.index).astype(str))
    key_counts = keys.value_counts()
    small_keys = key_counts[key_counts < n_splits].index.tolist()
    if small_keys:
        anchor_key = key_counts.idxmax()
        keys = keys.replace({key: anchor_key for key in small_keys})
    return keys


def build_cv_splits(y: np.ndarray, n_splits: int, seed: int) -> list[tuple[np.ndarray, np.ndarray]]:
    if MultilabelStratifiedKFold is not None:
        splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return list(splitter.split(np.zeros(len(y)), y))
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_keys = build_fold_keys(y, n_splits)
    return list(splitter.split(np.zeros(len(y)), fold_keys))




def tune_thresholds(
    probabilities: np.ndarray,
    y_true: np.ndarray,
    start: float = 0.01,
    stop: float = 0.99,
    step: float = 0.005,
    target_positive_rates: np.ndarray | None = None,
    rate_penalty: float = 0.0,
) -> list[float]:
    thresholds = []
    for label_idx in range(y_true.shape[1]):
        grid = np.arange(start, stop + 1e-9, step)
        quantile_grid = np.quantile(probabilities[:, label_idx], np.linspace(0.01, 0.99, 120))
        grid = np.unique(
            np.clip(
                np.concatenate([grid, quantile_grid, quantile_grid - 0.0025, quantile_grid + 0.0025, np.array([0.5])]),
                0.001, 0.999,
            )
        )
        best_threshold = 0.5
        best_score = -1.0
        target_rate = None if target_positive_rates is None else float(target_positive_rates[label_idx])
        rare_weight = 1.0 if target_rate is None else float(1.0 / np.sqrt(max(target_rate, 1e-4)))
        for threshold in grid:
            predictions = (probabilities[:, label_idx] >= threshold).astype(int)
            score = (predictions == y_true[:, label_idx]).mean()
            if target_positive_rates is not None and rate_penalty > 0:
                predicted_rate = predictions.mean()
                underprediction_gap = max(0.0, target_rate - float(predicted_rate))
                score -= rate_penalty * rare_weight * underprediction_gap
            if score > best_score:
                best_score = float(score)
                best_threshold = float(threshold)
        thresholds.append(best_threshold)
    return thresholds


def apply_thresholds(probabilities: np.ndarray, thresholds: list[float]) -> np.ndarray:
    return (probabilities >= np.asarray(thresholds)).astype(np.int64)


def refine_thresholds_with_global_shift(
    probabilities: np.ndarray,
    y_true: np.ndarray,
    base_thresholds: list[float],
    rate_penalty: float = 0.0,
    cardinality_penalty: float = 0.0,
    zero_penalty: float = 0.0,
    start: float = -0.08,
    stop: float = 0.04,
    step: float = 0.0025,
) -> tuple[list[float], float]:
    thresholds_array = np.asarray(base_thresholds, dtype=np.float64)
    if rate_penalty <= 0 and cardinality_penalty <= 0 and zero_penalty <= 0:

        rate_penalty = 0.0
        cardinality_penalty = 0.0
        zero_penalty = 0.0

    target_positive_rates = y_true.mean(axis=0)
    rare_weights = 1.0 / np.sqrt(np.maximum(target_positive_rates, 1e-4))
    target_multi_share = float((y_true.sum(axis=1) >= 2).mean())
    target_zero_share = float((y_true.sum(axis=1) == 0).mean())

    shift_grid = np.unique(
        np.clip(np.concatenate([np.arange(start, stop + 1e-9, step), np.array([0.0])]), -0.25, 0.25)
    )

    best_thresholds = thresholds_array.copy()
    best_shift = 0.0
    best_score = -1e18

    for shift in shift_grid:
        shifted_thresholds = np.clip(thresholds_array + float(shift), 0.001, 0.999)
        predictions = apply_thresholds(probabilities, shifted_thresholds.tolist())
        score = hamming_score(y_true, predictions)
        if rate_penalty > 0:
            predicted_rates = predictions.mean(axis=0)
            underprediction_gap = np.maximum(0.0, target_positive_rates - predicted_rates)
            score -= rate_penalty * float(np.mean(rare_weights * underprediction_gap))
        if cardinality_penalty > 0:
            predicted_multi_share = float((predictions.sum(axis=1) >= 2).mean())
            score -= cardinality_penalty * max(0.0, target_multi_share - predicted_multi_share)
        if zero_penalty > 0:
            predicted_zero_share = float((predictions.sum(axis=1) == 0).mean())
            score -= zero_penalty * max(0.0, predicted_zero_share - target_zero_share)
        if score > best_score + 1e-12 or (abs(score - best_score) <= 1e-12 and abs(float(shift)) < abs(best_shift)):
            best_score = float(score)
            best_shift = float(shift)
            best_thresholds = shifted_thresholds
    return best_thresholds.tolist(), best_shift


def tune_and_score(
    probabilities: np.ndarray,
    y_true: np.ndarray,
    threshold_rate_penalty: float = 0.0,
    threshold_cardinality_penalty: float = 0.0,
    threshold_zero_penalty: float = 0.0,
    threshold_global_shift_start: float = -0.08,
    threshold_global_shift_stop: float = 0.04,
    threshold_global_shift_step: float = 0.0025,
) -> dict:
    thresholds = tune_thresholds(
        probabilities, y_true,
        target_positive_rates=y_true.mean(axis=0),
        rate_penalty=threshold_rate_penalty,
    )
    thresholds, threshold_shift = refine_thresholds_with_global_shift(
        probabilities, y_true, thresholds,
        rate_penalty=threshold_rate_penalty,
        cardinality_penalty=threshold_cardinality_penalty,
        zero_penalty=threshold_zero_penalty,
        start=threshold_global_shift_start,
        stop=threshold_global_shift_stop,
        step=threshold_global_shift_step,
    )
    predictions = apply_thresholds(probabilities, thresholds)
    return {
        "thresholds": thresholds,
        "threshold_shift": threshold_shift,
        "predictions": predictions,
        "score": hamming_score(y_true, predictions),
    }


def tune_source_aware_thresholds(
    oof_probabilities: np.ndarray,
    y_true: np.ndarray,
    train_sources: np.ndarray,
    test_sources: np.ndarray,
    base_thresholds: list[float],
    shrinkage_alpha: float = 0.5,
    min_source_size: int = 200,
) -> dict[str, list[float]]:
    """
    Compute per-source thresholds with shrinkage toward global thresholds.
    For test sources not in train, fall back to global.
    Returns: dict mapping source-name -> per-class threshold list.
    """
    global_thresholds = np.asarray(base_thresholds, dtype=np.float64)
    per_source: dict[str, list[float]] = {}
    train_sources = np.asarray(train_sources).astype(str)
    test_unique = np.unique(np.asarray(test_sources).astype(str))

    for source_name in np.unique(np.concatenate([train_sources, test_unique])):
        mask = train_sources == source_name
        if mask.sum() < min_source_size:
            per_source[source_name] = global_thresholds.tolist()
            continue
        local_thresholds = tune_thresholds(
            oof_probabilities[mask], y_true[mask],
            target_positive_rates=y_true[mask].mean(axis=0),
            rate_penalty=0.0,
        )
        local_arr = np.asarray(local_thresholds, dtype=np.float64)

        size = float(mask.sum())
        local_weight = min(1.0, size / 5000.0) * shrinkage_alpha
        shrunk = (1.0 - local_weight) * global_thresholds + local_weight * local_arr
        per_source[source_name] = np.clip(shrunk, 0.001, 0.999).tolist()


    for src in test_unique:
        if src not in per_source:
            per_source[src] = global_thresholds.tolist()
    return per_source


def apply_per_source_thresholds(
    probabilities: np.ndarray,
    sources: np.ndarray,
    per_source_thresholds: dict[str, list[float]],
    default_thresholds: list[float],
) -> np.ndarray:
    sources = np.asarray(sources).astype(str)
    out = np.zeros_like(probabilities, dtype=np.int64)
    default_arr = np.asarray(default_thresholds)
    for i, src in enumerate(sources):
        thr = np.asarray(per_source_thresholds.get(src, default_arr))
        out[i] = (probabilities[i] >= thr).astype(np.int64)
    return out



def _stack_proba(estimator: OneVsRestClassifier, features) -> np.ndarray:
    return np.column_stack([est.predict_proba(features)[:, 1] for est in estimator.estimators_])


def run_sparse_view_cv(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    y: np.ndarray,
    splits: list[tuple[np.ndarray, np.ndarray]],
    text_column: str,
    word_vectorizer_config: dict,
    char_vectorizer_config: dict | None = None,
    c_value: float = 4.0,
    max_iter: int = 700,
    class_weight: str | dict | None = None,
    seed: int = 322,
    tag: str = "Sparse",
) -> dict[str, np.ndarray]:
    oof = np.zeros((len(train_frame), y.shape[1]), dtype=np.float32)
    test_probs = np.zeros((len(test_frame), y.shape[1]), dtype=np.float32)
    for fold_id, (train_idx, valid_idx) in enumerate(splits, start=1):
        train_part = train_frame.iloc[train_idx]
        valid_part = train_frame.iloc[valid_idx]
        word_vec = TfidfVectorizer(**word_vectorizer_config)
        train_word = word_vec.fit_transform(train_part[text_column])
        valid_word = word_vec.transform(valid_part[text_column])
        test_word = word_vec.transform(test_frame[text_column])
        if char_vectorizer_config is not None:
            char_vec = TfidfVectorizer(**char_vectorizer_config)
            train_char = char_vec.fit_transform(train_part[text_column])
            valid_char = char_vec.transform(valid_part[text_column])
            test_char = char_vec.transform(test_frame[text_column])
            train_mat = sparse.hstack([train_word, train_char], format="csr")
            valid_mat = sparse.hstack([valid_word, valid_char], format="csr")
            test_mat = sparse.hstack([test_word, test_char], format="csr")
        else:
            train_mat, valid_mat, test_mat = train_word, valid_word, test_word
        clf = OneVsRestClassifier(
            LogisticRegression(C=c_value, solver="liblinear", max_iter=max_iter,
                               class_weight=class_weight, random_state=seed),
            n_jobs=1,
        )
        clf.fit(train_mat, y[train_idx])
        fv = _stack_proba(clf, valid_mat)
        ft = _stack_proba(clf, test_mat)
        oof[valid_idx] = fv
        test_probs += ft / len(splits)
        s = hamming_score(y[valid_idx], (fv >= 0.5).astype(np.int64))
        print(f"[{tag}] Fold {fold_id}/{len(splits)} score@0.5 = {s:.6f}")
    return {"oof": oof, "test": test_probs}


def run_sparse_multi_view_cv(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    y: np.ndarray,
    n_splits: int = 5,
    seed: int = 322,
    c_value: float = 4.0,
    max_iter: int = 700,
    class_weight: str | dict | None = None,
    enabled_views: list[str] | None = None,
) -> dict[str, dict[str, np.ndarray]]:
    splits = build_cv_splits(y, n_splits, seed)
    view_configs = {
        "sparse_plain": {
            "text_column": "plain_text",
            "word": {"analyzer": "word", "ngram_range": (1, 2), "min_df": 3, "max_df": 0.995, "max_features": 120000, "sublinear_tf": True},
            "char": {"analyzer": "char_wb", "ngram_range": (3, 5), "min_df": 3, "max_features": 50000, "sublinear_tf": True},
        },
        "sparse_main": {
            "text_column": "combined_text",
            "word": {"analyzer": "word", "ngram_range": (1, 2), "min_df": 3, "max_df": 0.995, "max_features": 120000, "sublinear_tf": True},
            "char": {"analyzer": "char_wb", "ngram_range": (3, 5), "min_df": 3, "max_features": 50000, "sublinear_tf": True},
        },
        "sparse_lead": {
            "text_column": "plain_lead_text",
            "word": {"analyzer": "word", "ngram_range": (1, 2), "min_df": 3, "max_df": 0.995, "max_features": 90000, "sublinear_tf": True},
            "char": {"analyzer": "char_wb", "ngram_range": (3, 5), "min_df": 3, "max_features": 35000, "sublinear_tf": True},
        },
    }
    outputs: dict[str, dict[str, np.ndarray]] = {}
    for view_name, cfg in view_configs.items():
        if enabled_views is not None and view_name not in enabled_views:
            continue
        outputs[view_name] = run_sparse_view_cv(
            train_frame=train_frame, test_frame=test_frame, y=y, splits=splits,
            text_column=cfg["text_column"],
            word_vectorizer_config=cfg["word"], char_vectorizer_config=cfg["char"],
            c_value=c_value, max_iter=max_iter, class_weight=class_weight, seed=seed, tag=view_name,
        )
    return outputs




class _TransformerTextDataset(Dataset):
    def __init__(self, encodings: dict[str, list[list[int]]], labels: np.ndarray | None = None) -> None:
        self.encodings = encodings
        self.labels = labels

    def __len__(self) -> int:
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


def extract_transformer_embeddings(
    texts: list[str],
    model_name: str,
    batch_size: int = 32,
    max_length: int = 256,
    device: str = "auto",
    num_workers: int = 0,
    pooling: str = "mean",
) -> np.ndarray:
    from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding

    active = resolve_device(device)
    amp_dtype = pick_amp_dtype(active)
    use_amp = active.type == "cuda"
    pin = active.type == "cuda"
    print(f"[Embed] model={model_name}, device={active}, amp={use_amp}({amp_dtype})")
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = tok(texts, truncation=True, max_length=max_length, padding=False)
    collator = DataCollatorWithPadding(tokenizer=tok, pad_to_multiple_of=8 if active.type == "cuda" else None)
    loader = DataLoader(
        _TransformerTextDataset(enc, labels=None),
        batch_size=batch_size, shuffle=False, collate_fn=collator,
        num_workers=num_workers, pin_memory=pin,
    )
    model = AutoModel.from_pretrained(model_name).to(active)
    model.eval()
    pooled_out = []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(active, non_blocking=pin)
            mask = batch["attention_mask"].to(active, non_blocking=pin)
            with torch.autocast(device_type=active.type, dtype=amp_dtype, enabled=use_amp):
                hidden = model(input_ids=ids, attention_mask=mask).last_hidden_state
                if pooling == "cls":
                    pooled = hidden[:, 0]
                else:
                    m = mask.unsqueeze(-1)
                    pooled = (hidden * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)
            pooled_out.append(pooled.float().cpu().numpy())
    embeds = np.vstack(pooled_out)
    embeds = normalize(embeds, norm="l2").astype(np.float32)
    del model, loader
    gc.collect()
    if active.type == "cuda":
        torch.cuda.empty_cache()
    return embeds


def run_embedding_lgbm_cv(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    y: np.ndarray,
    model_name: str,
    text_column: str,
    n_splits: int = 5,
    seed: int = 322,
    batch_size: int = 32,
    max_length: int = 256,
    device: str = "auto",
    num_workers: int = 0,
    lgbm_params: dict | None = None,
    lgbm_rounds: int = 500,
    early_stopping: int = 40,
    extra_features_train: np.ndarray | None = None,
    extra_features_test: np.ndarray | None = None,
) -> dict[str, np.ndarray]:
    """
    DL → classical: build embeddings with a transformer (frozen),
    then train one LightGBM per label with proper OOF.
    """
    if not LIGHTGBM_AVAILABLE:
        raise RuntimeError("lightgbm is required for the EmbeddingGO branch")
    set_seed(seed)
    combined_texts = train_frame[text_column].tolist() + test_frame[text_column].tolist()
    embeds = extract_transformer_embeddings(
        combined_texts, model_name=model_name, batch_size=batch_size, max_length=max_length,
        device=device, num_workers=num_workers, pooling="mean",
    )
    train_embeds = embeds[: len(train_frame)]
    test_embeds = embeds[len(train_frame):]
    if extra_features_train is not None:
        train_embeds = np.hstack([train_embeds, extra_features_train.astype(np.float32)])
        test_embeds = np.hstack([test_embeds, extra_features_test.astype(np.float32)])

    splits = build_cv_splits(y, n_splits, seed)
    oof = np.zeros((len(train_frame), y.shape[1]), dtype=np.float32)
    test_probs = np.zeros((len(test_frame), y.shape[1]), dtype=np.float32)

    default_params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "min_data_in_leaf": 30,
        "lambda_l2": 1.0,
        "max_depth": -1,
        "verbosity": -1,
    }
    if lgbm_params:
        default_params.update(lgbm_params)

    for label_idx in range(y.shape[1]):
        label_oof = np.zeros(len(train_frame), dtype=np.float32)
        label_test = np.zeros(len(test_frame), dtype=np.float32)
        for fold_id, (train_idx, valid_idx) in enumerate(splits, start=1):
            params = dict(default_params)
            params["seed"] = seed + fold_id
            params["feature_fraction_seed"] = seed + fold_id
            params["bagging_seed"] = seed + fold_id
            train_set = lgb.Dataset(train_embeds[train_idx], label=y[train_idx, label_idx])
            valid_set = lgb.Dataset(train_embeds[valid_idx], label=y[valid_idx, label_idx], reference=train_set)
            booster = lgb.train(
                params, train_set, num_boost_round=lgbm_rounds,
                valid_sets=[valid_set],
                callbacks=[lgb.early_stopping(stopping_rounds=early_stopping, verbose=False),
                           lgb.log_evaluation(period=0)],
            )
            label_oof[valid_idx] = booster.predict(train_embeds[valid_idx], num_iteration=booster.best_iteration)
            label_test += booster.predict(test_embeds, num_iteration=booster.best_iteration) / len(splits)
        oof[:, label_idx] = label_oof
        test_probs[:, label_idx] = label_test
        s = hamming_score(y[:, label_idx:label_idx + 1], (label_oof.reshape(-1, 1) >= 0.5).astype(np.int64))
        print(f"[EmbedLGBM] label {label_idx} acc@0.5={s:.6f}")
    return {"oof": oof, "test": test_probs, "train_embeds": train_embeds, "test_embeds": test_embeds}



def run_mlm_domain_adaptation(
    texts: list[str],
    base_model_name: str,
    output_dir: str,
    epochs: int = 1,
    batch_size: int = 8,
    grad_accumulation_steps: int = 2,
    max_length: int = 384,
    learning_rate: float = 5e-5,
    warmup_ratio: float = 0.06,
    mlm_probability: float = 0.15,
    seed: int = 322,
    device: str = "auto",
    gradient_checkpointing: bool = True,
    num_workers: int = 0,
) -> str:
    """
    Continue MLM pre-training of `base_model_name` on `texts` (train+test combined).
    Saves adapted model to output_dir and returns the path.
    """
    from transformers import (
        AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling, get_scheduler,
    )

    set_seed(seed)
    active = resolve_device(device)
    amp_dtype = pick_amp_dtype(active)
    use_amp = active.type == "cuda"
    pin = active.type == "cuda"
    print(f"[MLM-DAPT] base={base_model_name}, device={active}, amp={use_amp}({amp_dtype})")
    tok = AutoTokenizer.from_pretrained(base_model_name)
    enc = tok(texts, truncation=True, max_length=max_length, padding=False)

    keep_idx = [i for i, ids in enumerate(enc["input_ids"]) if len(ids) > 16]
    enc = {k: [v[i] for i in keep_idx] for k, v in enc.items()}
    collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=mlm_probability)

    class _MLMDataset(Dataset):
        def __init__(self, encodings):
            self.encodings = encodings
        def __len__(self):
            return len(self.encodings["input_ids"])
        def __getitem__(self, i):
            return {"input_ids": self.encodings["input_ids"][i],
                    "attention_mask": self.encodings["attention_mask"][i]}

    loader = DataLoader(
        _MLMDataset(enc), batch_size=batch_size, shuffle=True,
        collate_fn=collator, num_workers=num_workers, pin_memory=pin,
    )
    model = AutoModelForMaskedLM.from_pretrained(base_model_name).to(active)
    if gradient_checkpointing:
        model.gradient_checkpointing_enable()
        if hasattr(model.config, "use_cache"):
            model.config.use_cache = False
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    total_steps = int(np.ceil(len(loader) / grad_accumulation_steps)) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_scheduler("linear", optimizer=optimizer,
                              num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

    model.train()
    t0 = time.time()
    for epoch_id in range(1, epochs + 1):
        optimizer.zero_grad()
        running = 0.0
        ncount = 0
        for step, batch in enumerate(loader, start=1):
            ids = batch["input_ids"].to(active, non_blocking=pin)
            mask = batch["attention_mask"].to(active, non_blocking=pin)
            labels = batch["labels"].to(active, non_blocking=pin)
            with torch.autocast(device_type=active.type, dtype=amp_dtype, enabled=use_amp):
                out = model(input_ids=ids, attention_mask=mask, labels=labels)
                loss = out.loss / grad_accumulation_steps
            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()
            if step % grad_accumulation_steps == 0 or step == len(loader):
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            running += float(loss.detach().float().cpu()) * grad_accumulation_steps
            ncount += 1
            if step % 200 == 0:
                elapsed = time.time() - t0
                print(f"[MLM-DAPT] epoch {epoch_id} step {step}/{len(loader)} loss={running/ncount:.4f} ({elapsed:.0f}s)")
        print(f"[MLM-DAPT] epoch {epoch_id} done, mean_loss={running/max(ncount,1):.4f}")

    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tok.save_pretrained(output_dir)
    del model, optimizer, scheduler, loader
    gc.collect()
    if active.type == "cuda":
        torch.cuda.empty_cache()
    print(f"[MLM-DAPT] saved adapted weights to {output_dir}")
    return output_dir




def tokenize_head_tail(
    texts: list[str],
    tokenizer,
    max_length: int = 384,
    head_ratio: float = 0.75,
) -> dict[str, list[list[int]]]:
    """
    Tokenize text and, if length > max_length, keep first head_tokens + last tail_tokens
    (so we don't drop the closing context — typical Kaggle long-doc trick).
    """
    raw = tokenizer(texts, truncation=False, padding=False, add_special_tokens=True)
    cls_id = tokenizer.cls_token_id if tokenizer.cls_token_id is not None else tokenizer.bos_token_id
    sep_id = tokenizer.sep_token_id if tokenizer.sep_token_id is not None else tokenizer.eos_token_id
    pad_special = [cls_id] if cls_id is not None else []
    end_special = [sep_id] if sep_id is not None else []
    body_budget = max_length - len(pad_special) - len(end_special)
    head_budget = int(body_budget * head_ratio)
    tail_budget = body_budget - head_budget

    new_ids = []
    new_mask = []
    for ids, mask in zip(raw["input_ids"], raw["attention_mask"]):
        if len(ids) <= max_length:
            new_ids.append(ids)
            new_mask.append(mask)
            continue

        body = ids
        if cls_id is not None and len(body) > 0 and body[0] == cls_id:
            body = body[1:]
        if sep_id is not None and len(body) > 0 and body[-1] == sep_id:
            body = body[:-1]
        if len(body) <= body_budget:
            kept = body
        else:
            kept = list(body[:head_budget]) + list(body[-tail_budget:])
        kept = pad_special + kept + end_special
        new_ids.append(kept)
        new_mask.append([1] * len(kept))
    return {"input_ids": new_ids, "attention_mask": new_mask}



class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg: float = 4.0, gamma_pos: float = 1.0, clip: float = 0.05, eps: float = 1e-8) -> None:
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        prob = torch.sigmoid(logits)
        pos_p = prob
        neg_p = 1.0 - prob
        if self.clip is not None and self.clip > 0:
            neg_p = (neg_p + self.clip).clamp(max=1.0)
        pos_loss = targets * torch.log(pos_p.clamp(min=self.eps))
        neg_loss = (1.0 - targets) * torch.log(neg_p.clamp(min=self.eps))
        loss = pos_loss + neg_loss
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt = pos_p * targets + neg_p * (1.0 - targets)
            gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
            loss = loss * torch.pow(1.0 - pt, gamma)
        return -loss.mean()




def build_llrd_param_groups(model: nn.Module, base_lr: float, decay: float = 0.95,
                            classifier_lr_multiplier: float = 5.0, weight_decay: float = 0.01) -> list[dict]:
    """
    Build parameter groups with layer-wise learning rate decay.
    Works for RoBERTa/BERT-family models with .roberta or .bert encoder.
    Falls back to single group if structure is unexpected.
    """
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    groups: list[dict] = []
    encoder = None
    for attr in ("roberta", "bert", "xlm_roberta", "deberta", "model", "transformer"):
        if hasattr(model, attr):
            sub = getattr(model, attr)
            if hasattr(sub, "encoder") and hasattr(sub.encoder, "layer"):
                encoder = sub
                break
    if encoder is None:

        groups.append({
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "lr": base_lr, "weight_decay": weight_decay,
        })
        groups.append({
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "lr": base_lr, "weight_decay": 0.0,
        })
        return groups

    n_layers = len(encoder.encoder.layer)

    embed_params_decay = []
    embed_params_nodecay = []
    if hasattr(encoder, "embeddings"):
        for n, p in encoder.embeddings.named_parameters():
            (embed_params_nodecay if any(nd in n for nd in no_decay) else embed_params_decay).append(p)
    embed_lr = base_lr * (decay ** (n_layers + 1))
    if embed_params_decay:
        groups.append({"params": embed_params_decay, "lr": embed_lr, "weight_decay": weight_decay})
    if embed_params_nodecay:
        groups.append({"params": embed_params_nodecay, "lr": embed_lr, "weight_decay": 0.0})

    for layer_id, layer in enumerate(encoder.encoder.layer):
        layer_lr = base_lr * (decay ** (n_layers - layer_id))
        decay_params = []
        nodecay_params = []
        for n, p in layer.named_parameters():
            (nodecay_params if any(nd in n for nd in no_decay) else decay_params).append(p)
        if decay_params:
            groups.append({"params": decay_params, "lr": layer_lr, "weight_decay": weight_decay})
        if nodecay_params:
            groups.append({"params": nodecay_params, "lr": layer_lr, "weight_decay": 0.0})


    encoder_param_ids = set(id(p) for p in encoder.parameters())
    head_decay = []
    head_nodecay = []
    for n, p in model.named_parameters():
        if id(p) in encoder_param_ids:
            continue
        (head_nodecay if any(nd in n for nd in no_decay) else head_decay).append(p)
    head_lr = base_lr * classifier_lr_multiplier
    if head_decay:
        groups.append({"params": head_decay, "lr": head_lr, "weight_decay": weight_decay})
    if head_nodecay:
        groups.append({"params": head_nodecay, "lr": head_lr, "weight_decay": 0.0})
    return groups



def run_transformer_cv(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    y: np.ndarray,
    model_name: str,
    text_column: str = "transformer_text",
    n_splits: int = 5,
    seed: int = 322,
    batch_size: int = 4,
    eval_batch_size: int = 16,
    epochs: int = 4,
    learning_rate: float = 1.5e-5,
    weight_decay: float = 0.01,
    patience: int = 2,
    max_length: int = 384,
    head_ratio: float = 0.75,
    grad_accumulation_steps: int = 4,
    device: str = "auto",
    use_pos_weight: bool = True,
    warmup_ratio: float = 0.08,
    max_grad_norm: float = 1.0,
    gradient_checkpointing: bool = True,
    loss_name: str = "asl",
    asl_gamma_neg: float = 4.0,
    asl_gamma_pos: float = 0.0,
    asl_clip: float = 0.05,
    num_workers: int = 0,
    selection_metric: str = "tuned_score",
    extra_seeds: tuple[int, ...] = (),
    use_llrd: bool = True,
    llrd_decay: float = 0.9,
    swa_last_k: int = 2,
) -> dict[str, np.ndarray]:
    """
    Multi-label fine-tuning with:
      - head+tail truncation for long docs
      - ASL or BCE loss with optional pos_weight
      - LLRD param groups
      - simple SWA-lite (avg of last `swa_last_k` epoch weights)
      - multi-seed bagging (extra_seeds): averages OOF/test predictions
    """
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, get_scheduler

    all_seeds = (seed,) + tuple(int(s) for s in extra_seeds)
    bag_oof = np.zeros((len(train_frame), y.shape[1]), dtype=np.float32)
    bag_test = np.zeros((len(test_frame), y.shape[1]), dtype=np.float32)

    for bag_id, bag_seed in enumerate(all_seeds, start=1):
        print(f"\n--- [Transformer] bag {bag_id}/{len(all_seeds)} seed={bag_seed} ---")
        set_seed(bag_seed)
        active = resolve_device(device)
        amp_dtype = pick_amp_dtype(active)
        use_amp = active.type == "cuda"
        pin = active.type == "cuda"
        splits = build_cv_splits(y, n_splits, bag_seed)

        tok = AutoTokenizer.from_pretrained(model_name)
        train_enc = tokenize_head_tail(train_frame[text_column].tolist(), tok, max_length=max_length, head_ratio=head_ratio)
        test_enc = tokenize_head_tail(test_frame[text_column].tolist(), tok, max_length=max_length, head_ratio=head_ratio)
        collator = DataCollatorWithPadding(tokenizer=tok, pad_to_multiple_of=8 if active.type == "cuda" else None)

        oof = np.zeros((len(train_frame), y.shape[1]), dtype=np.float32)
        test_probs = np.zeros((len(test_frame), y.shape[1]), dtype=np.float32)

        for fold_id, (train_idx, valid_idx) in enumerate(splits, start=1):
            fold_seed = bag_seed + 1000 * fold_id
            set_seed(fold_seed)
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name, num_labels=y.shape[1],
                problem_type="multi_label_classification",
                ignore_mismatched_sizes=True,
            ).to(active)
            if gradient_checkpointing:
                model.gradient_checkpointing_enable()
                if hasattr(model.config, "use_cache"):
                    model.config.use_cache = False

            if use_llrd:
                param_groups = build_llrd_param_groups(model, base_lr=learning_rate, decay=llrd_decay,
                                                       classifier_lr_multiplier=4.0, weight_decay=weight_decay)
                optimizer = torch.optim.AdamW(param_groups)
            else:
                optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

            if loss_name == "asl":
                criterion = AsymmetricLoss(gamma_neg=asl_gamma_neg, gamma_pos=asl_gamma_pos, clip=asl_clip)
            else:
                if use_pos_weight:
                    fy = y[train_idx]
                    pos = fy.sum(axis=0).astype(np.float32)
                    neg = len(fy) - pos
                    pw = neg / np.clip(pos, 1.0, None)
                    pw = np.clip(pw, 1.0, 8.0)
                    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, dtype=torch.float32, device=active))
                else:
                    criterion = nn.BCEWithLogitsLoss()

            scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

            train_e = {k: [train_enc[k][i] for i in train_idx] for k in train_enc}
            valid_e = {k: [train_enc[k][i] for i in valid_idx] for k in train_enc}

            train_loader = DataLoader(
                _TransformerTextDataset(train_e, y[train_idx].astype(np.float32)),
                batch_size=batch_size, shuffle=True, collate_fn=collator,
                num_workers=num_workers, pin_memory=pin,
            )
            valid_loader = DataLoader(
                _TransformerTextDataset(valid_e, y[valid_idx].astype(np.float32)),
                batch_size=eval_batch_size, shuffle=False, collate_fn=collator,
                num_workers=num_workers, pin_memory=pin,
            )
            test_loader = DataLoader(
                _TransformerTextDataset(test_enc, labels=None),
                batch_size=eval_batch_size, shuffle=False, collate_fn=collator,
                num_workers=num_workers, pin_memory=pin,
            )

            total_steps = int(np.ceil(len(train_loader) / grad_accumulation_steps)) * epochs
            warmup_steps = int(total_steps * warmup_ratio)
            scheduler = get_scheduler("linear", optimizer=optimizer,
                                      num_warmup_steps=warmup_steps, num_training_steps=total_steps)

            epoch_states = []
            epoch_scores = []
            patience_counter = 0
            best_score = -1.0
            best_loss = float("inf")

            for epoch_id in range(1, epochs + 1):
                model.train()
                optimizer.zero_grad()
                for step, batch in enumerate(train_loader, start=1):
                    ids = batch["input_ids"].to(active, non_blocking=pin)
                    mask = batch["attention_mask"].to(active, non_blocking=pin)
                    labels = batch["labels"].to(active, non_blocking=pin)
                    with torch.autocast(device_type=active.type, dtype=amp_dtype, enabled=use_amp):
                        logits = model(input_ids=ids, attention_mask=mask).logits
                        loss = criterion(logits, labels) / grad_accumulation_steps
                    if scaler.is_enabled():
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()
                    if step % grad_accumulation_steps == 0 or step == len(train_loader):
                        if scaler.is_enabled():
                            scaler.unscale_(optimizer)
                        if max_grad_norm and max_grad_norm > 0:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                        if scaler.is_enabled():
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad()


                model.eval()
                v_preds = []
                v_losses = []
                with torch.no_grad():
                    for batch in valid_loader:
                        labels = batch["labels"].to(active, non_blocking=pin)
                        with torch.autocast(device_type=active.type, dtype=amp_dtype, enabled=use_amp):
                            logits = model(
                                input_ids=batch["input_ids"].to(active, non_blocking=pin),
                                attention_mask=batch["attention_mask"].to(active, non_blocking=pin),
                            ).logits
                            v_loss = criterion(logits, labels)
                        v_losses.append(float(v_loss.detach().float().cpu()))
                        v_preds.append(torch.sigmoid(logits).float().cpu().numpy())
                v_probs = np.vstack(v_preds)
                v_loss_mean = float(np.mean(v_losses))
                v_thr = tune_thresholds(v_probs, y[valid_idx])
                v_score = hamming_score(y[valid_idx], apply_thresholds(v_probs, v_thr))
                s05 = hamming_score(y[valid_idx], (v_probs >= 0.5).astype(np.int64))
                print(f"[Trans] fold {fold_id}/{n_splits} epoch {epoch_id}: "
                      f"val_loss={v_loss_mean:.5f} s@0.5={s05:.5f} tuned={v_score:.5f}")


                state_cpu = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
                epoch_states.append(state_cpu)
                epoch_scores.append({"loss": v_loss_mean, "score": v_score, "s05": s05})

                if selection_metric == "tuned_score":
                    improved = v_score > best_score + 1e-9
                elif selection_metric == "score@0.5":
                    improved = s05 > best_score + 1e-9
                else:
                    improved = v_loss_mean < best_loss - 1e-9

                if improved:
                    best_score = max(best_score, v_score)
                    best_loss = min(best_loss, v_loss_mean)
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(f"[Trans] fold {fold_id}: early stop at epoch {epoch_id}")
                        break


            k_eff = min(swa_last_k, len(epoch_states))
            if selection_metric == "tuned_score":
                order = np.argsort([-e["score"] for e in epoch_scores])
            elif selection_metric == "score@0.5":
                order = np.argsort([-e["s05"] for e in epoch_scores])
            else:
                order = np.argsort([e["loss"] for e in epoch_scores])
            top_idx = order[:k_eff]
            ref = epoch_states[top_idx[0]]
            avg_state = {n: torch.zeros_like(ref[n], dtype=torch.float32) for n in ref}
            for i in top_idx:
                for n in avg_state:
                    avg_state[n] += epoch_states[i][n].float()
            for n in avg_state:
                avg_state[n] /= float(k_eff)

            cast_state = {n: avg_state[n].to(epoch_states[top_idx[0]][n].dtype) for n in avg_state}
            model.load_state_dict(cast_state)
            model.eval()


            v_preds = []
            with torch.no_grad():
                for batch in valid_loader:
                    with torch.autocast(device_type=active.type, dtype=amp_dtype, enabled=use_amp):
                        logits = model(
                            input_ids=batch["input_ids"].to(active, non_blocking=pin),
                            attention_mask=batch["attention_mask"].to(active, non_blocking=pin),
                        ).logits
                    v_preds.append(torch.sigmoid(logits).float().cpu().numpy())
            oof[valid_idx] = np.vstack(v_preds)

            t_preds = []
            with torch.no_grad():
                for batch in test_loader:
                    with torch.autocast(device_type=active.type, dtype=amp_dtype, enabled=use_amp):
                        logits = model(
                            input_ids=batch["input_ids"].to(active, non_blocking=pin),
                            attention_mask=batch["attention_mask"].to(active, non_blocking=pin),
                        ).logits
                    t_preds.append(torch.sigmoid(logits).float().cpu().numpy())
            test_probs += np.vstack(t_preds) / n_splits

            del model, optimizer, scheduler, criterion, scaler, train_loader, valid_loader, test_loader, epoch_states
            gc.collect()
            if active.type == "cuda":
                torch.cuda.empty_cache()

        bag_oof += oof / len(all_seeds)
        bag_test += test_probs / len(all_seeds)

    return {"oof": bag_oof, "test": bag_test}




def normalize_weights(weights: dict[str, float]) -> dict[str, float]:
    positive = {k: max(float(v), 0.0) for k, v in weights.items()}
    total = sum(positive.values())
    if total <= 0:
        raise ValueError("All weights are zero")
    return {k: v / total for k, v in positive.items()}


def blend_logit_space(component_probs: dict[str, np.ndarray], weights: dict[str, float], eps: float = 1e-5) -> np.ndarray:
    normed = normalize_weights(weights)
    first = next(iter(component_probs.values()))
    out = np.zeros_like(first, dtype=np.float64)
    for name, w in normed.items():
        p = np.clip(component_probs[name], eps, 1.0 - eps)
        out += w * np.log(p / (1.0 - p))
    return 1.0 / (1.0 + np.exp(-out)).astype(np.float32)


def optimize_blend_weights(
    component_oof: dict[str, np.ndarray],
    component_test: dict[str, np.ndarray],
    y_true: np.ndarray,
    n_restarts: int = 8,
    seed: int = 322,
    threshold_rate_penalty: float = 0.0,
) -> dict[str, Any]:
    """
    Use Nelder-Mead with simplex restarts to find weights maximizing OOF tuned hamming.
    """
    component_names = list(component_oof.keys())
    n = len(component_names)
    rng = np.random.RandomState(seed)

    def objective(raw_weights: np.ndarray) -> float:

        ex = np.exp(raw_weights - raw_weights.max())
        weights = ex / ex.sum()
        wdict = {name: float(w) for name, w in zip(component_names, weights)}
        blended = blend_logit_space(component_oof, wdict)
        res = tune_and_score(blended, y_true, threshold_rate_penalty=threshold_rate_penalty)
        return -res["score"]

    best_score = float("inf")
    best_weights = None
    for restart in range(n_restarts):
        if restart == 0:
            x0 = np.zeros(n, dtype=np.float64)
        else:
            x0 = rng.randn(n) * 0.5
        res = minimize(objective, x0, method="Nelder-Mead",
                       options={"maxiter": 200, "xatol": 1e-3, "fatol": 1e-5, "adaptive": True})
        if res.fun < best_score:
            best_score = float(res.fun)
            best_weights = res.x.copy()

    ex = np.exp(best_weights - best_weights.max())
    weights = ex / ex.sum()
    wdict = {name: float(w) for name, w in zip(component_names, weights)}
    blended_oof = blend_logit_space(component_oof, wdict)
    blended_test = blend_logit_space(component_test, wdict)
    return {
        "weights": wdict,
        "oof": blended_oof,
        "test": blended_test,
        "oof_score": -best_score,
    }


def build_meta_features(
    frame: pd.DataFrame,
    component_probs: dict[str, np.ndarray],
    component_names: list[str] | None = None,
) -> tuple[np.ndarray, list[str]]:
    if component_names is None:
        component_names = sorted(component_probs)
    blocks: list[np.ndarray] = []
    names: list[str] = []
    for name in component_names:
        p = component_probs[name].astype(np.float32)
        blocks.append(p)
        names.extend([f"{name}_lab{i}" for i in range(p.shape[1])])
        blocks.append(p.sum(axis=1, keepdims=True))
        names.append(f"{name}_sum")
        blocks.append(p.max(axis=1, keepdims=True))
        names.append(f"{name}_max")
    title_len = (np.log1p(frame["title_clean"].str.len().to_numpy(np.float32)) / 10.0).reshape(-1, 1)
    text_len = (np.log1p(frame["text_clean"].str.len().to_numpy(np.float32)) / 10.0).reshape(-1, 1)
    lead_len = (np.log1p(frame["plain_lead_text"].str.len().to_numpy(np.float32)) / 10.0).reshape(-1, 1)
    blocks.extend([title_len, text_len, lead_len])
    names.extend(["title_log_len", "text_log_len", "lead_log_len"])

    sources = frame["source"].astype(str).to_numpy()
    unique_sources = sorted(set(sources))
    for src in unique_sources:
        blocks.append((sources == src).astype(np.float32).reshape(-1, 1))
        names.append(f"source_{src}")
    return np.hstack(blocks).astype(np.float32), names


def run_meta_lgbm_stack(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    y: np.ndarray,
    component_oof: dict[str, np.ndarray],
    component_test: dict[str, np.ndarray],
    n_splits: int = 5,
    seed: int = 322,
    lgbm_rounds: int = 300,
    early_stopping: int = 30,
) -> dict[str, np.ndarray]:
    if not LIGHTGBM_AVAILABLE:
        raise RuntimeError("lightgbm is required for the meta-stack")
    component_names = sorted(component_oof)

    combined_sources = sorted(set(train_frame["source"].astype(str)).union(set(test_frame["source"].astype(str))))
    def featurize(frame, probs):
        blocks = []
        for name in component_names:
            p = probs[name].astype(np.float32)
            blocks.append(p)
            blocks.append(p.sum(axis=1, keepdims=True))
            blocks.append(p.max(axis=1, keepdims=True))
        title_len = (np.log1p(frame["title_clean"].str.len().to_numpy(np.float32)) / 10.0).reshape(-1, 1)
        text_len = (np.log1p(frame["text_clean"].str.len().to_numpy(np.float32)) / 10.0).reshape(-1, 1)
        lead_len = (np.log1p(frame["plain_lead_text"].str.len().to_numpy(np.float32)) / 10.0).reshape(-1, 1)
        blocks.extend([title_len, text_len, lead_len])
        srcs = frame["source"].astype(str).to_numpy()
        for src in combined_sources:
            blocks.append((srcs == src).astype(np.float32).reshape(-1, 1))
        return np.hstack(blocks).astype(np.float32)

    train_feats = featurize(train_frame, component_oof)
    test_feats = featurize(test_frame, component_test)
    splits = build_cv_splits(y, n_splits, seed + 101)
    oof = np.zeros((len(train_frame), y.shape[1]), dtype=np.float32)
    test_probs = np.zeros((len(test_frame), y.shape[1]), dtype=np.float32)

    default_params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "learning_rate": 0.04,
        "num_leaves": 31,
        "feature_fraction": 0.85,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "min_data_in_leaf": 30,
        "lambda_l2": 1.5,
        "verbosity": -1,
    }

    for label_idx in range(y.shape[1]):
        label_oof = np.zeros(len(train_frame), dtype=np.float32)
        label_test = np.zeros(len(test_frame), dtype=np.float32)
        for fold_id, (train_idx, valid_idx) in enumerate(splits, start=1):
            params = dict(default_params)
            params["seed"] = seed + fold_id
            ds_train = lgb.Dataset(train_feats[train_idx], label=y[train_idx, label_idx])
            ds_valid = lgb.Dataset(train_feats[valid_idx], label=y[valid_idx, label_idx], reference=ds_train)
            booster = lgb.train(
                params, ds_train, num_boost_round=lgbm_rounds,
                valid_sets=[ds_valid],
                callbacks=[lgb.early_stopping(stopping_rounds=early_stopping, verbose=False),
                           lgb.log_evaluation(period=0)],
            )
            label_oof[valid_idx] = booster.predict(train_feats[valid_idx], num_iteration=booster.best_iteration)
            label_test += booster.predict(test_feats, num_iteration=booster.best_iteration) / len(splits)
        oof[:, label_idx] = label_oof
        test_probs[:, label_idx] = label_test
        print(f"[MetaLGBM] label {label_idx} done")
    return {"oof": oof, "test": test_probs}




def compute_candidate_profile(
    probabilities: np.ndarray,
    y_true: np.ndarray,
    frame: pd.DataFrame,
    threshold_kwargs: dict | None = None,
) -> dict:
    threshold_kwargs = threshold_kwargs or {}
    result = tune_and_score(probabilities, y_true, **threshold_kwargs)
    predictions = result["predictions"]
    global_score = float(result["score"])
    global_loss = 1.0 - global_score

    source_losses = {}
    for src in sorted(frame["source"].astype(str).unique()):
        mask = frame["source"].astype(str).to_numpy() == src
        source_losses[src] = float(1.0 - hamming_score(y_true[mask], predictions[mask]))
    worst_source = max(source_losses.values()) if source_losses else global_loss

    recent_mask = frame["publication_dt"] >= pd.Timestamp("2020-07-01")
    recent_loss = float(1.0 - hamming_score(y_true[recent_mask], predictions[recent_mask])) if recent_mask.any() else global_loss

    text_words = frame["text_clean"].str.split().map(len).to_numpy()
    long_thr = float(np.quantile(text_words, 0.75))
    long_mask = text_words >= long_thr
    long_loss = float(1.0 - hamming_score(y_true[long_mask], predictions[long_mask])) if long_mask.any() else global_loss

    target_multi = float((y_true.sum(axis=1) >= 2).mean())
    pred_multi = float((predictions.sum(axis=1) >= 2).mean())
    target_zero = float((y_true.sum(axis=1) == 0).mean())
    pred_zero = float((predictions.sum(axis=1) == 0).mean())

    train_rates = y_true.mean(axis=0)
    pred_rates = predictions.mean(axis=0)
    rare_weights = 1.0 / np.sqrt(np.maximum(train_rates, 1e-4))
    rare_under = float(np.mean(rare_weights * np.maximum(0.0, train_rates - pred_rates)))

    composite = (
        global_loss
        + 0.22 * worst_source
        + 0.18 * recent_loss
        + 0.08 * long_loss
        + 0.28 * rare_under
        + 0.45 * max(0.0, target_multi - pred_multi)
        + 0.16 * max(0.0, pred_zero - target_zero)
        + 0.05 * abs(result["threshold_shift"])
    )

    return {
        "result": result,
        "global_score": global_score,
        "global_loss": global_loss,
        "worst_source_loss": worst_source,
        "recent_loss": recent_loss,
        "long_text_loss": long_loss,
        "target_multi_share": target_multi,
        "pred_multi_share": pred_multi,
        "target_zero_share": target_zero,
        "pred_zero_share": pred_zero,
        "rare_under": rare_under,
        "threshold_shift": float(result["threshold_shift"]),
        "composite_score": float(composite),
        "source_losses": source_losses,
    }


def build_submission(
    sample_submission: pd.DataFrame,
    test_ids: pd.Series,
    predicted_labels: np.ndarray,
) -> pd.DataFrame:
    submission = sample_submission.copy()
    submission["id"] = test_ids.to_numpy()
    submission["target"] = [format_target_list(row) for row in predicted_labels]
    return submission


print("[pipeline] module loaded. LightGBM available:", LIGHTGBM_AVAILABLE)

[pipeline] module loaded. LightGBM available: False


In [ ]:


RUN_MODE = 'full'  
STRATEGY = 'V10_MLM_DAPT_ruRoBERTa_HEAD_TAIL_PLUS_EMBED_LGBM'

THRESHOLD_KWARGS = {
    'threshold_rate_penalty': 0.008,
    'threshold_cardinality_penalty': 0.035,
    'threshold_zero_penalty': 0.015,
    'threshold_global_shift_start': -0.06,
    'threshold_global_shift_stop': 0.02,
    'threshold_global_shift_step': 0.0025,
}


SPARSE_CFG = {
    'n_splits': 5, 'seed': SEED, 'c_value': 4.0, 'max_iter': 700,
    'class_weight': 'balanced',
    'enabled_views': ['sparse_plain', 'sparse_main'],
    'manual_view_weights': {'sparse_plain': 0.7, 'sparse_main': 0.3},
}

EMBED_LGBM_CFG = {
    'model_name': 'intfloat/multilingual-e5-base',
    'fallback_model_name': 'cointegrated/rubert-tiny2',
    'text_column': 'e5_passage_text',
    'fallback_text_column': 'plain_transformer_text',
    'n_splits': 5,
    'seed': SEED,
    'batch_size': 32,
    'max_length': 384,
    'device': 'auto',
    'num_workers': 0,
    'lgbm_params': {
        'learning_rate': 0.05,
        'num_leaves': 63,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.85,
        'bagging_freq': 1,
        'min_data_in_leaf': 30,
        'lambda_l2': 1.0,
    },
    'lgbm_rounds': 800,
    'early_stopping': 60,
}


MLM_DAPT_CFG = {
    'enabled': True,
    'base_model_name': 'ai-forever/ruRoberta-large',
    'fallback_base_model_name': 'DeepPavlov/rubert-base-cased',
    'output_dir': './mlm_dapt_runtime',
    'epochs': 1,
    'batch_size': 4,
    'grad_accumulation_steps': 4,
    'max_length': 384,
    'learning_rate': 5e-5,
    'warmup_ratio': 0.06,
    'mlm_probability': 0.15,
    'seed': SEED,
    'gradient_checkpointing': True,
    'num_workers': 0,
}


MAIN_TRANSFORMER_CFG = {
    'model_name': None,  
    'text_column': 'plain_transformer_text',
    'n_splits': 5,
    'seed': SEED,
    'batch_size': 4,
    'eval_batch_size': 16,
    'epochs': 4,
    'learning_rate': 1.2e-5,
    'weight_decay': 0.01,
    'patience': 2,
    'max_length': 384,
    'head_ratio': 0.75,
    'grad_accumulation_steps': 4,
    'device': 'auto',
    'use_pos_weight': True,
    'warmup_ratio': 0.08,
    'max_grad_norm': 1.0,
    'gradient_checkpointing': True,
    'loss_name': 'asl',
    'asl_gamma_neg': 4.0,
    'asl_gamma_pos': 0.0,
    'asl_clip': 0.05,
    'num_workers': 0,
    'selection_metric': 'tuned_score',
    'extra_seeds': (339,),         
    'use_llrd': True,
    'llrd_decay': 0.9,
    'swa_last_k': 2,
}


DIVERSITY_TRANSFORMER_CFG = {
    'model_name': 'xlm-roberta-base',
    'text_column': 'plain_transformer_text',
    'n_splits': 5,
    'seed': 356,
    'batch_size': 8,
    'eval_batch_size': 32,
    'epochs': 4,
    'learning_rate': 1.8e-5,
    'weight_decay': 0.01,
    'patience': 2,
    'max_length': 320,
    'head_ratio': 0.75,
    'grad_accumulation_steps': 2,
    'device': 'auto',
    'use_pos_weight': True,
    'warmup_ratio': 0.08,
    'max_grad_norm': 1.0,
    'gradient_checkpointing': True,
    'loss_name': 'bce',
    'asl_gamma_neg': 4.0,
    'asl_gamma_pos': 0.0,
    'asl_clip': 0.05,
    'num_workers': 0,
    'selection_metric': 'tuned_score',
    'extra_seeds': (),
    'use_llrd': True,
    'llrd_decay': 0.9,
    'swa_last_k': 2,
}


if RUN_MODE == 'smoke':
    SPARSE_CFG['n_splits'] = 2
    SPARSE_CFG['max_iter'] = 200
    SPARSE_CFG['enabled_views'] = ['sparse_plain']
    SPARSE_CFG['manual_view_weights'] = {'sparse_plain': 1.0}

    EMBED_LGBM_CFG['model_name'] = 'cointegrated/rubert-tiny2'
    EMBED_LGBM_CFG['fallback_model_name'] = 'cointegrated/rubert-tiny2'
    EMBED_LGBM_CFG['text_column'] = 'plain_transformer_text'
    EMBED_LGBM_CFG['n_splits'] = 2
    EMBED_LGBM_CFG['batch_size'] = 16
    EMBED_LGBM_CFG['max_length'] = 128
    EMBED_LGBM_CFG['lgbm_rounds'] = 50
    EMBED_LGBM_CFG['early_stopping'] = 10
    EMBED_LGBM_CFG['device'] = 'cpu'

    MLM_DAPT_CFG['enabled'] = False
    MLM_DAPT_CFG['base_model_name'] = 'cointegrated/rubert-tiny2'

    MAIN_TRANSFORMER_CFG.update({
        'model_name': 'cointegrated/rubert-tiny2',
        'n_splits': 2, 'epochs': 1, 'batch_size': 8, 'eval_batch_size': 16,
        'max_length': 128, 'grad_accumulation_steps': 1, 'patience': 1,
        'extra_seeds': (), 'gradient_checkpointing': False, 'device': 'cpu',
        'loss_name': 'bce', 'use_llrd': False,
    })
    DIVERSITY_TRANSFORMER_CFG.update({
        'model_name': 'cointegrated/rubert-tiny2',
        'n_splits': 2, 'epochs': 1, 'batch_size': 8, 'eval_batch_size': 16,
        'max_length': 128, 'grad_accumulation_steps': 1, 'patience': 1,
        'gradient_checkpointing': False, 'device': 'cpu', 'use_llrd': False,
    })

print('RUN_MODE =', RUN_MODE)
print('STRATEGY =', STRATEGY)
print('Threshold kwargs:')
display(pd.Series(THRESHOLD_KWARGS).to_frame('value'))
print('Main transformer cfg (will be filled with MLM-adapted weights):')
display(pd.Series({k: v for k, v in MAIN_TRANSFORMER_CFG.items() if k != 'extra_seeds'}).to_frame('value'))
print('Diversity transformer cfg:')
display(pd.Series({k: v for k, v in DIVERSITY_TRANSFORMER_CFG.items() if k != 'extra_seeds'}).to_frame('value'))

RUN_MODE = full
STRATEGY = V10_MLM_DAPT_ruRoBERTa_HEAD_TAIL_PLUS_EMBED_LGBM
Threshold kwargs:


,value
threshold_rate_penalty,0.0080
threshold_cardinality_penalty,0.0350
threshold_zero_penalty,0.0150
threshold_global_shift_start,-0.0600
threshold_global_shift_stop,0.0200
threshold_global_shift_step,0.0025


Main transformer cfg (will be filled with MLM-adapted weights):


,value
model_name,None
text_column,plain_transformer_text
n_splits,5
seed,322
batch_size,4
eval_batch_size,16
epochs,4
learning_rate,0.000012
weight_decay,0.01
patience,2


Diversity transformer cfg:


,value
model_name,xlm-roberta-base
text_column,plain_transformer_text
n_splits,5
seed,356
batch_size,8
eval_batch_size,32
epochs,4
learning_rate,0.000018
weight_decay,0.01
patience,2


In [ ]:


train_df = pd.read_csv(TRAIN_PATH, sep='\t')
test_df = pd.read_csv(TEST_PATH, sep='\t')
sample_submission_df = pd.read_csv(SAMPLE_PATH)  

train_frame = prepare_frame(train_df)
test_frame = prepare_frame(test_df)
y = parse_targets(train_frame['target'])

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Sample submission shape:', sample_submission_df.shape)
print('Targets shape:', y.shape, '(should be (n_train, 5))')
print('First parsed target:', y[0].tolist())

Train shape: (16701, 6)
Test shape: (4969, 5)
Sample submission shape: (4969, 2)
Targets shape: (16701, 5) (should be (n_train, 5))
First parsed target: [0, 1, 0, 0, 0]


In [5]:
source_train = train_frame['source'].astype(str).value_counts().rename_axis('source').reset_index(name='train_rows')
source_test = test_frame['source'].astype(str).value_counts().rename_axis('source').reset_index(name='test_rows')
source_summary = source_train.merge(source_test, on='source', how='outer').fillna(0)
source_summary['seen_in_train'] = source_summary['train_rows'] > 0
source_summary['test_share'] = source_summary['test_rows'] / len(test_frame)
display(source_summary)

unseen_test_sources = source_summary.loc[~source_summary['seen_in_train'], 'source'].tolist()
unseen_test_share = float(source_summary.loc[~source_summary['seen_in_train'], 'test_share'].sum())
print(f'⚠ Unseen test sources: {unseen_test_sources}, total share of test: {unseen_test_share:.3f}')
print('This is a domain-shift problem. MLM-DAPT and diversity backbones (XLM-R) help here.')


label_rates = pd.DataFrame({
    'label': [f'label_{i}' for i in range(y.shape[1])],
    'positive_rate': y.mean(axis=0),
    'positive_count': y.sum(axis=0),
})
display(label_rates)


label_count_summary = pd.DataFrame([
    {'bucket': '0 labels', 'share': float((y.sum(axis=1) == 0).mean()), 'count': int((y.sum(axis=1) == 0).sum())},
    {'bucket': '1 label', 'share': float((y.sum(axis=1) == 1).mean()), 'count': int((y.sum(axis=1) == 1).sum())},
    {'bucket': '2 labels', 'share': float((y.sum(axis=1) == 2).mean()), 'count': int((y.sum(axis=1) == 2).sum())},
    {'bucket': '3+ labels', 'share': float((y.sum(axis=1) >= 3).mean()), 'count': int((y.sum(axis=1) >= 3).sum())},
])
display(label_count_summary)


per_source_rates = []
for src, sub in train_frame.groupby('source'):
    idx = sub.index.to_numpy()
    rates = y[idx].mean(axis=0)
    per_source_rates.append({'source': src, 'n_train': len(sub),
                             **{f'label_{i}': float(rates[i]) for i in range(y.shape[1])}})
display(pd.DataFrame(per_source_rates))


length_rows = []
for name, df in [('train', train_frame), ('test', test_frame)]:
    for src, sub in df.groupby('source'):
        L = sub['text'].astype(str).str.len()
        length_rows.append({
            'split': name, 'source': src, 'rows': len(sub),
            'len_q50': int(L.quantile(0.5)), 'len_q90': int(L.quantile(0.9)),
            'len_q99': int(L.quantile(0.99)),
        })
display(pd.DataFrame(length_rows))


stress_table = pd.DataFrame([
    {'metric': 'test unseen-source share', 'value': unseen_test_share},
    {'metric': 'test Aug-2020 share',
     'value': float((test_frame['publication_dt'] >= pd.Timestamp('2020-08-01')).mean())},
    {'metric': 'train Jul-2020+ share',
     'value': float((train_frame['publication_dt'] >= pd.Timestamp('2020-07-01')).mean())},
    {'metric': 'train all-zero target share',
     'value': float((y.sum(axis=1) == 0).mean())},
])
display(stress_table)

,source,train_rows,test_rows,seen_in_train,test_share
0,Novosti,12759.0,1996,True,0.401690
1,Spletnesti,0.0,867,False,0.174482
2,Svezhesti,3942.0,568,True,0.114309
3,Zholtosti,0.0,1538,False,0.309519


⚠ Unseen test sources: ['Spletnesti', 'Zholtosti'], total share of test: 0.484
This is a domain-shift problem. MLM-DAPT and diversity backbones (XLM-R) help here.


,label,positive_rate,positive_count
0,label_0,0.427819,7145
1,label_1,0.136279,2276
2,label_2,0.109934,1836
3,label_3,0.074427,1243
4,label_4,0.032573,544


,bucket,share,count
0,0 labels,0.337884,5643
1,1 label,0.551404,9209
2,2 labels,0.102629,1714
3,3+ labels,0.008083,135


,source,n_train,label_0,label_1,label_2,label_3,label_4
0,Novosti,12759,0.431382,0.136296,0.109413,0.080884,0.013794
1,Svezhesti,3942,0.416286,0.136225,0.111618,0.053526,0.093354


,split,source,rows,len_q50,len_q90,len_q99
0,train,Novosti,12759,1834,3312,8436
1,train,Svezhesti,3942,1619,2321,4557
2,test,Novosti,1996,1409,5329,9097
3,test,Spletnesti,867,4577,7261,13032
4,test,Svezhesti,568,1200,1786,5121
5,test,Zholtosti,1538,831,7114,23087


,metric,value
0,test unseen-source share,0.484001
1,test Aug-2020 share,0.526062
2,train Jul-2020+ share,0.139333
3,train all-zero target share,0.337884


In [ ]:


set_seed(SEED)
sparse_outputs = run_sparse_multi_view_cv(
    train_frame=train_frame, test_frame=test_frame, y=y,
    n_splits=SPARSE_CFG['n_splits'], seed=SPARSE_CFG['seed'],
    c_value=SPARSE_CFG['c_value'], max_iter=SPARSE_CFG['max_iter'],
    class_weight=SPARSE_CFG['class_weight'],
    enabled_views=SPARSE_CFG['enabled_views'],
)


sparse_blend_weights = SPARSE_CFG['manual_view_weights']
sparse_oof = np.zeros_like(next(iter(sparse_outputs.values()))['oof'])
sparse_test = np.zeros_like(next(iter(sparse_outputs.values()))['test'])
total_w = sum(sparse_blend_weights.get(name, 0.0) for name in sparse_outputs)
for name in sparse_outputs:
    w = sparse_blend_weights.get(name, 0.0) / total_w if total_w > 0 else 1.0 / len(sparse_outputs)
    sparse_oof += w * sparse_outputs[name]['oof']
    sparse_test += w * sparse_outputs[name]['test']
sparse_oof = sparse_oof.astype(np.float32)
sparse_test = sparse_test.astype(np.float32)

sparse_profile = compute_candidate_profile(sparse_oof, y, train_frame, THRESHOLD_KWARGS)
print(f"[SparseBlend] OOF tuned hamming = {sparse_profile['global_score']:.6f}")
print(f"[SparseBlend] threshold shift = {sparse_profile['threshold_shift']:+.4f}")
print(f"[SparseBlend] worst source loss = {sparse_profile['worst_source_loss']:.6f}, "
      f"recent loss = {sparse_profile['recent_loss']:.6f}")

[sparse_plain] Fold 1/5 score@0.5 = 0.943952
[sparse_plain] Fold 2/5 score@0.5 = 0.942874


In [ ]:
set_seed(SEED)
embed_model = EMBED_LGBM_CFG['model_name']
embed_text_col = EMBED_LGBM_CFG['text_column']
embed_outputs = None
try:
    embed_outputs = run_embedding_lgbm_cv(
        train_frame=train_frame, test_frame=test_frame, y=y,
        model_name=embed_model, text_column=embed_text_col,
        n_splits=EMBED_LGBM_CFG['n_splits'], seed=EMBED_LGBM_CFG['seed'],
        batch_size=EMBED_LGBM_CFG['batch_size'], max_length=EMBED_LGBM_CFG['max_length'],
        device=EMBED_LGBM_CFG['device'], num_workers=EMBED_LGBM_CFG['num_workers'],
        lgbm_params=EMBED_LGBM_CFG['lgbm_params'],
        lgbm_rounds=EMBED_LGBM_CFG['lgbm_rounds'],
        early_stopping=EMBED_LGBM_CFG['early_stopping'],
    )
except Exception as exc:
    print(f"[EmbedLGBM] primary model {embed_model} failed: {type(exc).__name__}: {exc}")
    fb_model = EMBED_LGBM_CFG['fallback_model_name']
    fb_text = EMBED_LGBM_CFG['fallback_text_column']
    print(f"[EmbedLGBM] falling back to {fb_model} on text column {fb_text}")
    embed_outputs = run_embedding_lgbm_cv(
        train_frame=train_frame, test_frame=test_frame, y=y,
        model_name=fb_model, text_column=fb_text,
        n_splits=EMBED_LGBM_CFG['n_splits'], seed=EMBED_LGBM_CFG['seed'],
        batch_size=EMBED_LGBM_CFG['batch_size'], max_length=EMBED_LGBM_CFG['max_length'],
        device=EMBED_LGBM_CFG['device'], num_workers=EMBED_LGBM_CFG['num_workers'],
        lgbm_params=EMBED_LGBM_CFG['lgbm_params'],
        lgbm_rounds=EMBED_LGBM_CFG['lgbm_rounds'],
        early_stopping=EMBED_LGBM_CFG['early_stopping'],
    )

embed_oof = embed_outputs['oof']
embed_test = embed_outputs['test']
embed_profile = compute_candidate_profile(embed_oof, y, train_frame, THRESHOLD_KWARGS)
print(f"[EmbedLGBM] OOF tuned hamming = {embed_profile['global_score']:.6f}")
print(f"[EmbedLGBM] threshold shift = {embed_profile['threshold_shift']:+.4f}")

In [ ]:
set_seed(SEED)
main_model_for_finetune = None
if MLM_DAPT_CFG['enabled']:
    try:
        mlm_corpus = (
            train_frame['plain_transformer_text'].tolist()
            + test_frame['plain_transformer_text'].tolist()
        )
        main_model_for_finetune = run_mlm_domain_adaptation(
            texts=mlm_corpus,
            base_model_name=MLM_DAPT_CFG['base_model_name'],
            output_dir=MLM_DAPT_CFG['output_dir'],
            epochs=MLM_DAPT_CFG['epochs'],
            batch_size=MLM_DAPT_CFG['batch_size'],
            grad_accumulation_steps=MLM_DAPT_CFG['grad_accumulation_steps'],
            max_length=MLM_DAPT_CFG['max_length'],
            learning_rate=MLM_DAPT_CFG['learning_rate'],
            warmup_ratio=MLM_DAPT_CFG['warmup_ratio'],
            mlm_probability=MLM_DAPT_CFG['mlm_probability'],
            seed=MLM_DAPT_CFG['seed'],
            device='auto',
            gradient_checkpointing=MLM_DAPT_CFG['gradient_checkpointing'],
            num_workers=MLM_DAPT_CFG['num_workers'],
        )
    except Exception as exc:
        print(f"[MLM-DAPT] failed with primary base {MLM_DAPT_CFG['base_model_name']}: "
              f"{type(exc).__name__}: {exc}")
        fb = MLM_DAPT_CFG['fallback_base_model_name']
        print(f"[MLM-DAPT] retrying with fallback base {fb}")
        try:
            main_model_for_finetune = run_mlm_domain_adaptation(
                texts=mlm_corpus,
                base_model_name=fb,
                output_dir=MLM_DAPT_CFG['output_dir'] + '_fallback',
                epochs=MLM_DAPT_CFG['epochs'],
                batch_size=MLM_DAPT_CFG['batch_size'] * 2,
                grad_accumulation_steps=max(1, MLM_DAPT_CFG['grad_accumulation_steps'] // 2),
                max_length=MLM_DAPT_CFG['max_length'],
                learning_rate=MLM_DAPT_CFG['learning_rate'],
                warmup_ratio=MLM_DAPT_CFG['warmup_ratio'],
                mlm_probability=MLM_DAPT_CFG['mlm_probability'],
                seed=MLM_DAPT_CFG['seed'],
                device='auto',
                gradient_checkpointing=MLM_DAPT_CFG['gradient_checkpointing'],
                num_workers=MLM_DAPT_CFG['num_workers'],
            )
        except Exception as exc2:
            print(f"[MLM-DAPT] fallback also failed: {type(exc2).__name__}: {exc2}")
            main_model_for_finetune = None


if main_model_for_finetune is None:

    main_model_for_finetune = MLM_DAPT_CFG['base_model_name']
    print(f"[MainDL] using base model without MLM-DAPT: {main_model_for_finetune}")


if MAIN_TRANSFORMER_CFG['model_name'] is None:
    MAIN_TRANSFORMER_CFG['model_name'] = main_model_for_finetune

main_cfg_for_run = dict(MAIN_TRANSFORMER_CFG)
print(f"[MainDL] starting fine-tune with model = {main_cfg_for_run['model_name']}")
try:
    main_outputs = run_transformer_cv(
        train_frame=train_frame, test_frame=test_frame, y=y,
        **main_cfg_for_run,
    )
except torch.cuda.OutOfMemoryError as exc:
    print(f"[MainDL] CUDA OOM with batch={main_cfg_for_run['batch_size']}; "
          f"retrying with batch=1, grad_accum={main_cfg_for_run['grad_accumulation_steps'] * main_cfg_for_run['batch_size']}")
    torch.cuda.empty_cache()
    gc.collect()
    main_cfg_for_run['grad_accumulation_steps'] = main_cfg_for_run['grad_accumulation_steps'] * main_cfg_for_run['batch_size']
    main_cfg_for_run['batch_size'] = 1
    main_cfg_for_run['eval_batch_size'] = max(2, main_cfg_for_run['eval_batch_size'] // 4)
    main_outputs = run_transformer_cv(
        train_frame=train_frame, test_frame=test_frame, y=y,
        **main_cfg_for_run,
    )
except Exception as exc:
    print(f"[MainDL] FAIL with model {main_cfg_for_run['model_name']}: {type(exc).__name__}: {exc}")
    fb_name = MLM_DAPT_CFG['fallback_base_model_name']
    print(f"[MainDL] retrying with fallback model {fb_name} (smaller, base size)")
    main_cfg_for_run['model_name'] = fb_name
    main_cfg_for_run['batch_size'] = 8
    main_cfg_for_run['eval_batch_size'] = 32
    main_cfg_for_run['grad_accumulation_steps'] = 2
    main_cfg_for_run['gradient_checkpointing'] = False
    main_outputs = run_transformer_cv(
        train_frame=train_frame, test_frame=test_frame, y=y,
        **main_cfg_for_run,
    )

main_oof = main_outputs['oof']
main_test = main_outputs['test']
main_profile = compute_candidate_profile(main_oof, y, train_frame, THRESHOLD_KWARGS)
print(f"[MainDL] OOF tuned hamming = {main_profile['global_score']:.6f}")
print(f"[MainDL] threshold shift = {main_profile['threshold_shift']:+.4f}")
print(f"[MainDL] worst source loss = {main_profile['worst_source_loss']:.6f}")


In [ ]:
set_seed(DIVERSITY_TRANSFORMER_CFG['seed'])
div_cfg = dict(DIVERSITY_TRANSFORMER_CFG)
print(f"[DivDL] starting fine-tune with model = {div_cfg['model_name']}")
try:
    div_outputs = run_transformer_cv(
        train_frame=train_frame, test_frame=test_frame, y=y,
        **div_cfg,
    )
except torch.cuda.OutOfMemoryError:
    print(f"[DivDL] CUDA OOM; halving batch and doubling grad_accum")
    torch.cuda.empty_cache(); gc.collect()
    div_cfg['grad_accumulation_steps'] *= 2
    div_cfg['batch_size'] = max(1, div_cfg['batch_size'] // 2)
    div_outputs = run_transformer_cv(
        train_frame=train_frame, test_frame=test_frame, y=y, **div_cfg,
    )
except Exception as exc:
    print(f"[DivDL] FAIL with {div_cfg['model_name']}: {type(exc).__name__}: {exc}")
    print(f"[DivDL] falling back to rubert-base-cased")
    div_cfg['model_name'] = 'DeepPavlov/rubert-base-cased'
    div_outputs = run_transformer_cv(
        train_frame=train_frame, test_frame=test_frame, y=y, **div_cfg,
    )

div_oof = div_outputs['oof']
div_test = div_outputs['test']
div_profile = compute_candidate_profile(div_oof, y, train_frame, THRESHOLD_KWARGS)
print(f"[DivDL] OOF tuned hamming = {div_profile['global_score']:.6f}")
print(f"[DivDL] threshold shift = {div_profile['threshold_shift']:+.4f}")
print(f"[DivDL] worst source loss = {div_profile['worst_source_loss']:.6f}")

In [ ]:
branch_summary_rows = []
branch_oof_components = {}
branch_test_components = {}
for name, prof, oof_p, test_p in [
    ('sparse', sparse_profile, sparse_oof, sparse_test),
    ('embed_lgbm', embed_profile, embed_oof, embed_test),
    ('main_dl', main_profile, main_oof, main_test),
    ('diversity_dl', div_profile, div_oof, div_test),
]:
    branch_oof_components[name] = oof_p
    branch_test_components[name] = test_p
    branch_summary_rows.append({
        'branch': name,
        'global_score': prof['global_score'],
        'worst_source_loss': prof['worst_source_loss'],
        'recent_loss': prof['recent_loss'],
        'long_text_loss': prof['long_text_loss'],
        'threshold_shift': prof['threshold_shift'],
        'composite_score': prof['composite_score'],
    })
branch_summary = pd.DataFrame(branch_summary_rows).sort_values('composite_score').reset_index(drop=True)
display(branch_summary)

best_single_branch_name = str(branch_summary.iloc[0]['branch'])
print(f'Best single branch by composite: {best_single_branch_name}')
print(f'  → OOF tuned hamming: {branch_summary.iloc[0]["global_score"]:.6f}')

In [ ]:
print('--- Optimizing logit-space blend weights ---')
blend_result = optimize_blend_weights(
    component_oof=branch_oof_components,
    component_test=branch_test_components,
    y_true=y, n_restarts=10, seed=SEED,
    threshold_rate_penalty=THRESHOLD_KWARGS['threshold_rate_penalty'],
)
blend_oof = blend_result['oof']
blend_test = blend_result['test']
print('Optimized blend weights:')
display(pd.Series(blend_result['weights']).to_frame('weight').sort_values('weight', ascending=False))
blend_profile = compute_candidate_profile(blend_oof, y, train_frame, THRESHOLD_KWARGS)
print(f"[Blend] OOF tuned hamming = {blend_profile['global_score']:.6f} (raw search obj = {blend_result['oof_score']:.6f})")

print('--- Training LightGBM meta-stack ---')
stack_outputs = run_meta_lgbm_stack(
    train_frame=train_frame, test_frame=test_frame, y=y,
    component_oof=branch_oof_components, component_test=branch_test_components,
    n_splits=SPARSE_CFG['n_splits'], seed=SEED,
    lgbm_rounds=400, early_stopping=40,
)
stack_oof = stack_outputs['oof']
stack_test = stack_outputs['test']
stack_profile = compute_candidate_profile(stack_oof, y, train_frame, THRESHOLD_KWARGS)
print(f"[Stack] OOF tuned hamming = {stack_profile['global_score']:.6f}")

ensemble_summary = pd.DataFrame([
    {'ensemble': 'logit_blend', **{k: blend_profile[k] for k in
        ['global_score', 'worst_source_loss', 'recent_loss', 'long_text_loss',
         'threshold_shift', 'composite_score']}},
    {'ensemble': 'meta_lgbm_stack', **{k: stack_profile[k] for k in
        ['global_score', 'worst_source_loss', 'recent_loss', 'long_text_loss',
         'threshold_shift', 'composite_score']}},
]).sort_values('composite_score').reset_index(drop=True)
display(ensemble_summary)

In [ ]:
candidate_pool = {
    'best_single': {
        'name': best_single_branch_name,
        'oof': branch_oof_components[best_single_branch_name],
        'test': branch_test_components[best_single_branch_name],
        'profile': next(p for p in [sparse_profile, embed_profile, main_profile, div_profile]
                        if p['composite_score'] == branch_summary[branch_summary['branch'] == best_single_branch_name]['composite_score'].iloc[0]),
    },
    'logit_blend': {'name': 'logit_blend', 'oof': blend_oof, 'test': blend_test, 'profile': blend_profile},
    'meta_stack': {'name': 'meta_lgbm_stack', 'oof': stack_oof, 'test': stack_test, 'profile': stack_profile},
}

candidate_table = pd.DataFrame([
    {'candidate': k, 'global_score': v['profile']['global_score'],
     'worst_source_loss': v['profile']['worst_source_loss'],
     'recent_loss': v['profile']['recent_loss'],
     'long_text_loss': v['profile']['long_text_loss'],
     'composite_score': v['profile']['composite_score']}
    for k, v in candidate_pool.items()
]).sort_values('composite_score').reset_index(drop=True)
display(candidate_table)

def is_ensemble_clearly_better(ens_profile: dict, base_profile: dict) -> bool:
    return (
        ens_profile['global_score'] >= base_profile['global_score'] + 0.0005
        and ens_profile['worst_source_loss'] <= base_profile['worst_source_loss'] + 0.001
        and ens_profile['recent_loss'] <= base_profile['recent_loss'] + 0.001
        and abs(ens_profile['threshold_shift']) <= max(0.02, abs(base_profile['threshold_shift']) + 0.02)
        and ens_profile['composite_score'] < base_profile['composite_score'] - 0.0008
    )

base = candidate_pool['best_single']
blend_cand = candidate_pool['logit_blend']
stack_cand = candidate_pool['meta_stack']


chosen = base
chosen_name = f"best_single({best_single_branch_name})"
for cand_name, cand in [('logit_blend', blend_cand), ('meta_stack', stack_cand)]:
    if is_ensemble_clearly_better(cand['profile'], base['profile']):
        if cand['profile']['composite_score'] < chosen['profile']['composite_score'] - 1e-12:
            chosen = cand
            chosen_name = cand_name

print(f"Final candidate selected: {chosen_name}")
print(f"  OOF tuned hamming: {chosen['profile']['global_score']:.6f}")
print(f"  Composite score:   {chosen['profile']['composite_score']:.6f}")
print(f"  Threshold shift:   {chosen['profile']['threshold_shift']:+.4f}")

final_oof = chosen['oof']
final_test = chosen['test']
final_profile = chosen['profile']

In [ ]:
global_result = final_profile['result']
global_thresholds = global_result['thresholds']
print('Global per-class thresholds:', [round(t, 4) for t in global_thresholds])


per_source_thr = tune_source_aware_thresholds(
    oof_probabilities=final_oof, y_true=y,
    train_sources=train_frame['source'].astype(str).to_numpy(),
    test_sources=test_frame['source'].astype(str).to_numpy(),
    base_thresholds=global_thresholds,
    shrinkage_alpha=0.5, min_source_size=300,
)
print('Per-source thresholds (after shrinkage):')
display(pd.DataFrame(per_source_thr).T.rename(columns={i: f'label_{i}' for i in range(y.shape[1])}))


oof_train_sources = train_frame['source'].astype(str).to_numpy()
per_source_oof_pred = apply_per_source_thresholds(final_oof, oof_train_sources, per_source_thr, global_thresholds)
global_oof_pred = apply_thresholds(final_oof, global_thresholds)
per_source_score = hamming_score(y, per_source_oof_pred)
global_score = hamming_score(y, global_oof_pred)
print(f"OOF hamming — global: {global_score:.6f} vs per-source-shrunk: {per_source_score:.6f}")

USE_PER_SOURCE = per_source_score >= global_score + 1e-5
print(f"Use per-source thresholds: {USE_PER_SOURCE}")


if USE_PER_SOURCE:
    final_test_labels = apply_per_source_thresholds(
        final_test, test_frame['source'].astype(str).to_numpy(),
        per_source_thr, global_thresholds,
    )
else:
    final_test_labels = apply_thresholds(final_test, global_thresholds)


submission = build_submission(sample_submission_df, test_df['id'], final_test_labels)
submission.to_csv(PRIMARY_SUBMISSION_PATH, index=False)
print(f"Saved submission to: {PRIMARY_SUBMISSION_PATH.resolve()}")
print(f"Rows: {len(submission)}; fingerprint: {compute_prediction_fingerprint(final_test_labels)}")


print('Positive label counts in test predictions:', final_test_labels.sum(axis=0).tolist())
print('Distribution of label-count per row:')
print(pd.Series(final_test_labels.sum(axis=1)).value_counts().sort_index().to_dict())
display(submission.head(10))

In [ ]:
if USE_PER_SOURCE:
    oof_pred_labels = apply_per_source_thresholds(
        final_oof, oof_train_sources, per_source_thr, global_thresholds,
    )
else:
    oof_pred_labels = apply_thresholds(final_oof, global_thresholds)

slice_rows = []
for src in sorted(train_frame['source'].astype(str).unique()):
    mask = train_frame['source'].astype(str).to_numpy() == src
    slice_rows.append({
        'slice': f'source={src}', 'rows': int(mask.sum()),
        'hamming_loss': float(1.0 - hamming_score(y[mask], oof_pred_labels[mask])),
    })
recent_mask = train_frame['publication_dt'] >= pd.Timestamp('2020-07-01')
slice_rows.append({
    'slice': 'month>=2020-07', 'rows': int(recent_mask.sum()),
    'hamming_loss': float(1.0 - hamming_score(y[recent_mask], oof_pred_labels[recent_mask])),
})
text_words = train_frame['text_clean'].str.split().map(len).to_numpy()
long_thr = float(np.quantile(text_words, 0.75))
long_mask = text_words >= long_thr
slice_rows.append({
    'slice': 'text_words>=q75', 'rows': int(long_mask.sum()),
    'hamming_loss': float(1.0 - hamming_score(y[long_mask], oof_pred_labels[long_mask])),
})
display(pd.DataFrame(slice_rows))


true_counts = y.sum(axis=1)
oof_counts = oof_pred_labels.sum(axis=1)
test_counts = final_test_labels.sum(axis=1)
display(pd.DataFrame([
    {'bucket': '0 labels',
     'train_share': float((true_counts == 0).mean()),
     'oof_share': float((oof_counts == 0).mean()),
     'test_share': float((test_counts == 0).mean())},
    {'bucket': '1 label',
     'train_share': float((true_counts == 1).mean()),
     'oof_share': float((oof_counts == 1).mean()),
     'test_share': float((test_counts == 1).mean())},
    {'bucket': '2+ labels',
     'train_share': float((true_counts >= 2).mean()),
     'oof_share': float((oof_counts >= 2).mean()),
     'test_share': float((test_counts >= 2).mean())},
]))

display(pd.DataFrame({
    'label': [f'label_{i}' for i in range(y.shape[1])],
    'train_positive_rate': y.mean(axis=0),
    'oof_positive_rate': oof_pred_labels.mean(axis=0),
    'test_positive_rate': final_test_labels.mean(axis=0),
}))

print('Final summary:')
print(f'  Strategy:        {STRATEGY}')
print(f'  Final candidate: {chosen_name}')
print(f'  OOF tuned hamming:        {final_profile["global_score"]:.6f}')
print(f'  OOF threshold shift:      {final_profile["threshold_shift"]:+.4f}')
print(f'  Submission path:          {PRIMARY_SUBMISSION_PATH.resolve()}')
print(f'  Submission fingerprint:   {compute_prediction_fingerprint(final_test_labels)}')